# City Property Tax Comparisons

This notebook downloads Utah property tax rate data, extracts it from PDF, and stores it in a SQLite database for analysis and comparison.

In [ ]:
from pathlib import Path
import requests
from pdfminer.high_level import extract_text_to_fp
from pdfminer.layout import LAParams
import pandas as pd
import sqlite3
import re
import io

# Setup paths
data_dir = Path("data")
pdf_path = data_dir / "taxarearates2025.pdf"

In [ ]:
# Parse tax rate PDF using pdfminer.six spatial layout analysis.
#
# Each page has a fixed column structure:
#   x < 380:  codes, names, tax area headers, totals  (left column)
#   x ≈ 410:  certified rates
#   x ≈ 463:  proposed rates
#   x ≈ 518:  adopted rates
#
# Strategy: for each page, collect all text boxes sorted by vertical position.
# For each tax area block, match left-column rows to rate columns by y-coordinate.

from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextBox
import re

print("Parsing tax data from PDF using pdfminer.six spatial layout...")

def parse_pdf_spatial(pdf_path):
    RATE_RE   = re.compile(r'^\d+\.\d+$')
    CODE_RE   = re.compile(r'^\d{4}$')
    COUNTY_RE = re.compile(r'^[A-Z][A-Z\s]+COUNTY$')
    SKIP_RE   = re.compile(
        r"^(UTAH STATE TAX COMMISSION|PROPERTY TAX DIVISION|"
        r"2025 Tax Rates by Tax Area|Auditor'?s?|Certified|Proposed|"
        r"Final|Adopted|Tax Rate|Rate|January|Report 600|Page \d+ of \d+)$"
    )

    # Column x-boundaries (from layout analysis)
    # Left content column: x0 < 380
    # Cert column:  380 <= x0 < 440
    # Prop column:  440 <= x0 < 495
    # Adopted col:  495 <= x0
    CERT_MIN, CERT_MAX   = 380, 440
    PROP_MIN, PROP_MAX   = 440, 495
    ADOPT_MIN            = 495

    records      = []
    parse_errors = []
    current_county   = None
    current_tax_area = None
    pending_entities = []  # list of (entity_code, entity_name, y_center)

    for page_num, page_layout in enumerate(extract_pages(pdf_path), 1):
        # Collect all text boxes on this page with their positions
        boxes = []
        for element in page_layout:
            if isinstance(element, LTTextBox):
                text = element.get_text().strip()
                if not text:
                    continue
                y_center = (element.y0 + element.y1) / 2
                boxes.append({
                    'x0': element.x0,
                    'y0': element.y0,
                    'y1': element.y1,
                    'y_center': y_center,
                    'text': text,
                })

        # Sort top-to-bottom
        boxes.sort(key=lambda b: -b['y_center'])

        # Build rate lookup: y_center -> {cert, prop, adopted}
        # Multiple rate values can be in one box (newline-separated)
        # We need to split them and assign to sub-rows by y position
        # For simplicity: collect rate boxes per column, split by newlines
        # into individual values, then match by rank within each tax area block.

        # Separate left-column content from rate columns
        left_boxes = [b for b in boxes if b['x0'] < CERT_MIN]
        cert_boxes = [b for b in boxes if CERT_MIN <= b['x0'] < CERT_MAX]
        prop_boxes = [b for b in boxes if PROP_MIN <= b['x0'] < PROP_MAX]
        adopt_boxes= [b for b in boxes if b['x0'] >= ADOPT_MIN]

        # Flatten rate boxes into ordered lists of values (top to bottom)
        def flatten_rate_box(box_list):
            vals = []
            for b in sorted(box_list, key=lambda x: -x['y_center']):
                for line in b['text'].split('\n'):
                    line = line.strip()
                    if RATE_RE.match(line):
                        vals.append((b['y_center'], line))
            return vals

        cert_vals  = flatten_rate_box(cert_boxes)
        prop_vals  = flatten_rate_box(prop_boxes)
        adopt_vals = flatten_rate_box(adopt_boxes)

        # Process left column to identify tax area boundaries
        # Each tax area block: header → (code,name) rows → Total line
        # We track which rates belong to each area by counting entities
        rate_pos = 0  # current position in cert/prop/adopt lists

        for b in left_boxes:
            text = b['text']
            # Could be multi-line (codes block or names block)
            for line in text.split('\n'):
                line = line.strip()
                if not line or SKIP_RE.match(line):
                    continue
                if COUNTY_RE.match(line):
                    current_county = line
                    continue
                if line.startswith('Tax Area '):
                    current_tax_area = line[len('Tax Area '):]
                    pending_entities = []
                    continue
                if line.startswith('Total for Tax Area '):
                    # Flush: assign next N cert/prop/adopted rates to pending entities
                    n = len(pending_entities)
                    if n > 0:
                        c_vals = cert_vals[rate_pos:rate_pos+n]
                        p_vals = prop_vals[rate_pos:rate_pos+n]
                        a_vals = adopt_vals[rate_pos:rate_pos+n]

                        if len(c_vals) < n or len(p_vals) < n or len(a_vals) < n:
                            parse_errors.append(
                                f"Page {page_num} Tax area {current_tax_area}: "
                                f"need {n} rates, got cert={len(c_vals)} prop={len(p_vals)} adopt={len(a_vals)}"
                            )
                        else:
                            for i, (code, name) in enumerate(pending_entities):
                                records.append({
                                    'county':         current_county,
                                    'tax_area':       current_tax_area,
                                    'entity_code':    code,
                                    'entity_name':    name,
                                    'auditor_rate':   float(c_vals[i][1]),
                                    'certified_rate': float(p_vals[i][1]),
                                    'adopted_rate':   float(a_vals[i][1]),
                                })
                        rate_pos += n
                    pending_entities = []
                    continue
                if CODE_RE.match(line) and current_tax_area:
                    pending_entities.append((line, ''))
                    continue
                # Name line: assign to most recent entity with empty name
                if current_tax_area and pending_entities:
                    for idx in range(len(pending_entities)-1, -1, -1):
                        if pending_entities[idx][1] == '':
                            code = pending_entities[idx][0]
                            pending_entities[idx] = (code, line)
                            break

    return records, parse_errors

records, parse_errors = parse_pdf_spatial(pdf_path)

print(f"Extracted {len(records)} records")

if parse_errors:
    print(f"\n*** {len(parse_errors)} parse errors: ***")
    for e in parse_errors[:20]:
        print(f"  {e}")
else:
    print("No parse errors detected.")

if records:
    combined_df = pd.DataFrame(records)
    print(f"\nColumns: {list(combined_df.columns)}")
    print(f"\nUnique tax areas: {combined_df['tax_area'].nunique()}")
    print(f"Unique counties: {combined_df['county'].nunique()}")
    print("\nFirst 20 rows:")
    display(combined_df.head(20))
else:
    print("No records extracted")
    combined_df = pd.DataFrame()

In [ ]:
# (This cell intentionally left blank - old pdfplumber parser replaced by cell above)

Parsing tax data from PDF...
PDF has 505 pages
Extracted 14443 records
No header/footer mismatches detected.

Columns: ['county', 'tax_area', 'entity_code', 'entity_name', 'auditor_rate', 'certified_rate', 'adopted_rate', 'page']

Unique tax areas: 996

First 20 rows:


,county,tax_area,entity_code,entity_name,auditor_rate,certified_rate,adopted_rate,page
0,BEAVER COUNTY,001 - 0000,1010,BEAVER,0.001462,0.001462,0.001462,1
1,BEAVER COUNTY,001 - 0000,1015,MULTICOUNTY ASSESSING,0.000014,0.000014,0.000014,1
2,BEAVER COUNTY,001 - 0000,1020,COUNTY ASSESSING,0.000294,0.000294,0.000294,1
3,BEAVER COUNTY,001 - 0000,2010,BEAVER,0.005190,0.005190,0.005190,1
4,BEAVER COUNTY,001 - 0000,4010,BEAVER COUNTY SPECIAL,0.000375,0.000375,0.000375,1
5,BEAVER COUNTY,001 - 0005,1010,BEAVER,0.001462,0.001462,0.001462,1
6,BEAVER COUNTY,001 - 0005,1015,MULTICOUNTY ASSESSING,0.000014,0.000014,0.000014,1
7,BEAVER COUNTY,001 - 0005,1020,COUNTY ASSESSING,0.000294,0.000294,0.000294,1
8,BEAVER COUNTY,001 - 0005,2010,BEAVER,0.005190,0.005190,0.005190,1
9,BEAVER COUNTY,001 - 0005,4010,BEAVER COUNTY SPECIAL,0.000375,0.000375,0.000375,1


In [30]:
# Save to CSV as backup
if not combined_df.empty:
    csv_path = data_dir / "tax_rates_parsed.csv"
    combined_df.to_csv(csv_path, index=False)
    print(f"Data saved to {csv_path}")

Data saved to data/tax_rates_parsed.csv


In [31]:
# Create SQLite database
db_path = data_dir / "property_tax.db"
conn = sqlite3.connect(db_path)

print(f"Creating database at {db_path}")

# Store the parsed data
if not combined_df.empty:
    combined_df.to_sql('tax_rates', conn, if_exists='replace', index=False)
    print("Data stored in 'tax_rates' table")
    
    # Verify the data
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM tax_rates")
    row_count = cursor.fetchone()[0]
    print(f"Total records in database: {row_count}")
    
    # Show table schema
    cursor.execute("PRAGMA table_info(tax_rates)")
    print("\nTable schema:")
    for col in cursor.fetchall():
        print(f"  {col[1]} ({col[2]})")
    
    # Show unique counties
    cursor.execute("SELECT DISTINCT county FROM tax_rates ORDER BY county")
    counties = [row[0] for row in cursor.fetchall()]
    print(f"\nCounties found: {len(counties)}")
    print(counties)
else:
    print("No data to store")

conn.close()
print("\nDatabase setup complete")

Creating database at data/property_tax.db
Data stored in 'tax_rates' table
Total records in database: 14443

Table schema:
  county (TEXT)
  tax_area (TEXT)
  entity_code (TEXT)
  entity_name (TEXT)
  auditor_rate (REAL)
  certified_rate (REAL)
  adopted_rate (REAL)
  page (INTEGER)

Counties found: 29
['BEAVER COUNTY', 'BOX ELDER COUNTY', 'CACHE COUNTY', 'CARBON COUNTY', 'DAGGETT COUNTY', 'DAVIS COUNTY', 'DUCHESNE COUNTY', 'EMERY COUNTY', 'GARFIELD COUNTY', 'GRAND COUNTY', 'IRON COUNTY', 'JUAB COUNTY', 'KANE COUNTY', 'MILLARD COUNTY', 'MORGAN COUNTY', 'PIUTE COUNTY', 'RICH COUNTY', 'SALT LAKE COUNTY', 'SAN JUAN COUNTY', 'SANPETE COUNTY', 'SEVIER COUNTY', 'SUMMIT COUNTY', 'TOOELE COUNTY', 'UINTAH COUNTY', 'UTAH COUNTY', 'WASATCH COUNTY', 'WASHINGTON COUNTY', 'WAYNE COUNTY', 'WEBER COUNTY']

Database setup complete


In [32]:
# Load municipal boundaries data
municipal_csv_path = data_dir / "UtahMunicipalBoundaries_221481185616367289.csv"

print(f"Loading municipal boundaries from {municipal_csv_path}")
municipal_df = pd.read_csv(municipal_csv_path)

print(f"Loaded {len(municipal_df)} municipalities")
print(f"\nColumns: {list(municipal_df.columns)}")
print("\nFirst 5 rows:")
display(municipal_df.head())

Loading municipal boundaries from data/UtahMunicipalBoundaries_221481185616367289.csv
Loaded 261 municipalities

Columns: ['OBJECTID', 'COUNTYNBR', 'NAME', 'COUNTYSEAT', 'SHORTDESC', 'UPDATED', 'FIPS', 'ENTITYNBR', 'SALESTAXID', 'IMSCOLOR', 'MINNAME', 'POPLASTCENSUS', 'POPLASTESTIMATE', 'UGRCODE', 'GNIS', 'GlobalID', 'Shape__Area', 'Shape__Length']

First 5 rows:


,OBJECTID,COUNTYNBR,NAME,COUNTYSEAT,SHORTDESC,UPDATED,FIPS,ENTITYNBR,SALESTAXID,IMSCOLOR,MINNAME,POPLASTCENSUS,POPLASTESTIMATE,UGRCODE,GNIS,GlobalID,Shape__Area,Shape__Length
0,1,3,Newton,0.0,NEWTON,10/20/2020 12:00:00 AM,54550.0,3100.0,47.0,2,,789.0,815.0,NWT,1430705.0,773ccaed-22b4-4237-ba15-17315a94a7d5,4.009109e+06,8948.894827
1,2,12,Eureka,0.0,EUREKA,NaN,24080.0,3010.0,9.0,2,,662.0,657.0,EUR,1437974.0,3ef04b24-1a05-4355-a900-d8c59ff118ae,6.387834e+06,12832.366127
2,3,29,Huntsville,0.0,HUNTSVILLE,5/2/2024 12:00:00 AM,37060.0,3030.0,19.0,4,Huntsville,573.0,593.0,HVL,1428949.0,f0274f4c-4dba-4475-97bb-329012e9c691,1.433332e+07,24584.998375
3,4,27,Springdale,0.0,SPRINGDALE,NaN,71840.0,3100.0,23.0,2,Springdale,514.0,576.0,SPD,1432867.0,9d93a698-af3a-48d3-9a9e-d49b1a39b74c,1.878263e+07,23966.386568
4,5,23,Grantsville,0.0,GRANTSVILLE,2/8/2022 12:00:00 AM,31120.0,3010.0,23.0,2,,12617.0,14417.0,GRA,1428338.0,d6bb102d-36fa-415b-947f-959c60413b85,1.835223e+08,112936.761973


In [33]:
# Transform municipal data
print("Transforming municipal data...")

# Add computed columns
municipal_df['area_sq_miles'] = municipal_df['Shape__Area'] / 2589988  # Convert sq meters to sq miles
municipal_df['normalized_name'] = municipal_df['NAME'].str.upper().str.replace(' CITY', '').str.replace(' TOWN', '').str.strip()

# Handle population - use estimate if available, otherwise census
municipal_df['population'] = municipal_df['POPLASTESTIMATE'].fillna(municipal_df['POPLASTCENSUS'])

# Select relevant columns for database
municipal_clean = municipal_df[[
    'OBJECTID', 'COUNTYNBR', 'NAME', 'SHORTDESC', 'ENTITYNBR', 
    'UGRCODE', 'population', 'Shape__Area', 'area_sq_miles', 'normalized_name'
]].copy()

print(f"Transformed data shape: {municipal_clean.shape}")
print("\nSample of transformed data:")
display(municipal_clean.head(10))

Transforming municipal data...
Transformed data shape: (261, 10)

Sample of transformed data:


,OBJECTID,COUNTYNBR,NAME,SHORTDESC,ENTITYNBR,UGRCODE,population,Shape__Area,area_sq_miles,normalized_name
0,1,3,Newton,NEWTON,3100.0,NWT,815.0,4.009109e+06,1.547926,NEWTON
1,2,12,Eureka,EUREKA,3010.0,EUR,657.0,6.387834e+06,2.466357,EUREKA
2,3,29,Huntsville,HUNTSVILLE,3030.0,HVL,593.0,1.433332e+07,5.534126,HUNTSVILLE
3,4,27,Springdale,SPRINGDALE,3100.0,SPD,576.0,1.878263e+07,7.252013,SPRINGDALE
4,5,23,Grantsville,GRANTSVILLE,3010.0,GRA,14417.0,1.835223e+08,70.858377,GRANTSVILLE
5,6,18,Bluffdale,BLUFFDALE (SL CO),3020.0,BDL,19080.0,4.671062e+07,18.035073,BLUFFDALE
6,7,19,Blanding,BLANDING,3010.0,BLA,3229.0,5.471993e+07,21.127483,BLANDING
7,8,18,Herriman,HERRIMAN TOWN,3035.0,HER,59179.0,1.036384e+08,40.015006,HERRIMAN
8,9,12,Levan,LEVAN,3020.0,LEV,892.0,3.482854e+06,1.344738,LEVAN
9,10,9,Panguitch,PANGUITCH,3070.0,PAN,1785.0,1.345872e+07,5.196440,PANGUITCH


In [34]:
# Load municipal data into SQLite
conn = sqlite3.connect(db_path)

print("Loading municipal data into database...")
municipal_clean.to_sql('municipal_boundaries', conn, if_exists='replace', index=False)

# Verify the data
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM municipal_boundaries")
row_count = cursor.fetchone()[0]
print(f"Total municipal records in database: {row_count}")

# Show table schema
cursor.execute("PRAGMA table_info(municipal_boundaries)")
print("\nTable schema:")
for col in cursor.fetchall():
    print(f"  {col[1]} ({col[2]})")

conn.close()
print("\nMunicipal data loaded successfully")

Loading municipal data into database...
Total municipal records in database: 261

Table schema:
  OBJECTID (INTEGER)
  COUNTYNBR (INTEGER)
  NAME (TEXT)
  SHORTDESC (TEXT)
  ENTITYNBR (REAL)
  UGRCODE (TEXT)
  population (REAL)
  Shape__Area (REAL)
  area_sq_miles (REAL)
  normalized_name (TEXT)

Municipal data loaded successfully


In [35]:
# Create county number-to-name mapping
conn = sqlite3.connect(db_path)

# Extract county names from tax data and create mapping
cursor = conn.cursor()
cursor.execute("SELECT DISTINCT county FROM tax_rates ORDER BY county")
counties = cursor.fetchall()

# Create mapping table
cursor.execute("DROP TABLE IF EXISTS county_mapping")
cursor.execute("""
CREATE TABLE county_mapping (
    county_nbr INTEGER PRIMARY KEY,
    county_name TEXT UNIQUE
)
""")

# County number to name mapping (based on Utah county codes)
county_map = {
    1: 'BEAVER COUNTY',
    2: 'BOX ELDER COUNTY', 
    3: 'CACHE COUNTY',
    4: 'CARBON COUNTY',
    5: 'DAGGETT COUNTY',
    6: 'DAVIS COUNTY',
    7: 'DUCHESNE COUNTY',
    8: 'EMERY COUNTY',
    9: 'GARFIELD COUNTY',
    10: 'GRAND COUNTY',
    11: 'IRON COUNTY',
    12: 'JUAB COUNTY',
    13: 'KANE COUNTY',
    14: 'MILLARD COUNTY',
    15: 'MORGAN COUNTY',
    16: 'PIUTE COUNTY',
    17: 'RICH COUNTY',
    18: 'SALT LAKE COUNTY',
    19: 'SAN JUAN COUNTY',
    20: 'SANPETE COUNTY',
    21: 'SEVIER COUNTY',
    22: 'SUMMIT COUNTY',
    23: 'TOOELE COUNTY',
    24: 'UINTAH COUNTY',
    25: 'UTAH COUNTY',
    26: 'WASATCH COUNTY',
    27: 'WASHINGTON COUNTY',
    28: 'WAYNE COUNTY',
    29: 'WEBER COUNTY'
}

# Insert mapping data
for nbr, name in county_map.items():
    cursor.execute("INSERT INTO county_mapping (county_nbr, county_name) VALUES (?, ?)", (nbr, name))

conn.commit()

# Verify mapping
cursor.execute("SELECT * FROM county_mapping ORDER BY county_nbr")
mapping_df = pd.read_sql_query("SELECT * FROM county_mapping ORDER BY county_nbr", conn)
print("County mapping table:")
display(mapping_df)

conn.close()
print("\nCounty mapping table created successfully")

County mapping table:


,county_nbr,county_name
0,1,BEAVER COUNTY
1,2,BOX ELDER COUNTY
2,3,CACHE COUNTY
3,4,CARBON COUNTY
4,5,DAGGETT COUNTY
5,6,DAVIS COUNTY
6,7,DUCHESNE COUNTY
7,8,EMERY COUNTY
8,9,GARFIELD COUNTY
9,10,GRAND COUNTY



County mapping table created successfully


In [36]:
# Add indexes for performance
conn = sqlite3.connect(db_path)

print("Adding indexes for performance...")

cursor = conn.cursor()

# Index on entity codes for joining
cursor.execute("CREATE INDEX IF NOT EXISTS idx_municipal_entity ON municipal_boundaries(ENTITYNBR)")
print("Created index on ENTITYNBR")

# Index on normalized names for alternative matching
cursor.execute("CREATE INDEX IF NOT EXISTS idx_municipal_name ON municipal_boundaries(normalized_name)")
print("Created index on normalized_name")

# Index on county number for joining with county mapping
cursor.execute("CREATE INDEX IF NOT EXISTS idx_municipal_county ON municipal_boundaries(COUNTYNBR)")
print("Created index on COUNTYNBR")

# Index on tax entity codes
cursor.execute("CREATE INDEX IF NOT EXISTS idx_tax_entity ON tax_rates(entity_code)")
print("Created index on tax_rates.entity_code")

# Index on tax county names
cursor.execute("CREATE INDEX IF NOT EXISTS idx_tax_county ON tax_rates(county)")
print("Created index on tax_rates.county")

conn.commit()
conn.close()
print("\nAll indexes created successfully")

Adding indexes for performance...
Created index on ENTITYNBR
Created index on normalized_name
Created index on COUNTYNBR
Created index on tax_rates.entity_code
Created index on tax_rates.county

All indexes created successfully


In [37]:
# Test queries removed - they produced incorrect results due to Cartesian product joins

Testing join by normalized entity names...
Join by normalized name (West Haven):


,county,entity_name,entity_code,municipal_name,area_sq_miles,population,adopted_rate
0,WEBER COUNTY,WEST HAVEN CITY,3130,West Haven,19.067111,22395.0,0.0
1,WEBER COUNTY,WEST HAVEN CITY,3130,West Haven,19.067111,22395.0,0.0
2,WEBER COUNTY,WEST HAVEN CITY,3130,West Haven,19.067111,22395.0,0.0
3,WEBER COUNTY,WEST HAVEN CITY,3130,West Haven,19.067111,22395.0,0.0
4,WEBER COUNTY,WEST HAVEN CITY,3130,West Haven,19.067111,22395.0,0.0
5,WEBER COUNTY,WEST HAVEN CITY,3130,West Haven,19.067111,22395.0,0.0
6,WEBER COUNTY,WEST HAVEN CITY,3130,West Haven,19.067111,22395.0,0.0
7,WEBER COUNTY,WEST HAVEN CITY,3130,West Haven,19.067111,22395.0,0.0
8,WEBER COUNTY,WEST HAVEN CITY,3130,West Haven,19.067111,22395.0,0.0
9,WEBER COUNTY,WEST HAVEN CITY,3130,West Haven,19.067111,22395.0,0.0



Testing county-level summary with municipal data...
County summary with municipal data:


,county_name,municipality_count,total_population,total_area_sq_miles,avg_tax_rate
0,SALT LAKE COUNTY,23,4.605937e+09,3.128143e+06,0.001055
1,UTAH COUNTY,27,8.139907e+08,7.077058e+05,0.001358
2,WEBER COUNTY,16,6.777017e+08,1.210792e+06,0.001075
3,DAVIS COUNTY,15,5.229056e+08,2.980105e+05,0.000907
4,WASHINGTON COUNTY,15,1.055800e+08,2.759243e+05,0.000982
5,BOX ELDER COUNTY,16,3.352592e+07,1.562467e+05,0.001203
6,TOOELE COUNTY,8,2.709669e+07,1.243408e+05,0.001778
7,CACHE COUNTY,19,2.434505e+07,6.391132e+04,0.001007
8,WASATCH COUNTY,9,1.130422e+07,4.083431e+04,0.001322
9,IRON COUNTY,6,9.436256e+06,1.751620e+04,0.001361



Join tests completed successfully


In [38]:
# Analyze multi-county cities
conn = sqlite3.connect(db_path)

print("Analyzing cities that span multiple counties...")
query = """
SELECT 
    NAME,
    COUNTYNBR,
    (SELECT county_name FROM county_mapping WHERE county_nbr = m.COUNTYNBR) as county_name,
    area_sq_miles,
    population,
    ROUND(area_sq_miles * 100.0 / SUM(area_sq_miles) OVER (PARTITION BY NAME), 2) as pct_of_total
FROM municipal_boundaries m
WHERE NAME IN (
    SELECT NAME 
    FROM municipal_boundaries 
    GROUP BY NAME 
    HAVING COUNT(*) > 1
)
ORDER BY NAME, area_sq_miles DESC
"""
multi_county_cities = pd.read_sql_query(query, conn)
print("Multi-county cities:")
display(multi_county_cities)

# Summary statistics
print("\nSummary of multi-county cities:")
for city in multi_county_cities['NAME'].unique():
    city_data = multi_county_cities[multi_county_cities['NAME'] == city]
    total_area = city_data['area_sq_miles'].sum()
    total_pop = city_data['population'].iloc[0]  # Same across entries
    print(f"\n{city}:")
    print(f"  Total area: {total_area:.2f} sq mi")
    print(f"  Population: {int(total_pop):,}")
    for _, row in city_data.iterrows():
        print(f"  {row['county_name']}: {row['area_sq_miles']:.2f} sq mi ({row['pct_of_total']}%)")

conn.close()
print("\nNote: Areas represent county-specific portions, not total city area.")

Analyzing cities that span multiple counties...
Multi-county cities:


,NAME,COUNTYNBR,county_name,area_sq_miles,population,pct_of_total
0,Bluffdale,18,SALT LAKE COUNTY,18.035073,19080.0,93.90
1,Bluffdale,25,UTAH COUNTY,1.171328,19080.0,6.10
2,Draper,18,SALT LAKE COUNTY,38.047134,50731.0,77.42
3,Draper,25,UTAH COUNTY,11.097566,50731.0,22.58
4,Park City,22,SUMMIT COUNTY,37.994962,8374.0,98.04
5,Park City,26,WASATCH COUNTY,0.758740,8374.0,1.96
6,Santaquin,25,UTAH COUNTY,17.706123,16898.0,98.11
7,Santaquin,12,JUAB COUNTY,0.340731,16898.0,1.89



Summary of multi-county cities:

Bluffdale:
  Total area: 19.21 sq mi
  Population: 19,080
  SALT LAKE COUNTY: 18.04 sq mi (93.9%)
  UTAH COUNTY: 1.17 sq mi (6.1%)

Draper:
  Total area: 49.14 sq mi
  Population: 50,731
  SALT LAKE COUNTY: 38.05 sq mi (77.42%)
  UTAH COUNTY: 11.10 sq mi (22.58%)

Park City:
  Total area: 38.75 sq mi
  Population: 8,374
  SUMMIT COUNTY: 37.99 sq mi (98.04%)
  WASATCH COUNTY: 0.76 sq mi (1.96%)

Santaquin:
  Total area: 18.05 sq mi
  Population: 16,898
  UTAH COUNTY: 17.71 sq mi (98.11%)
  JUAB COUNTY: 0.34 sq mi (1.89%)

Note: Areas represent county-specific portions, not total city area.


In [39]:
# Analyze entity code structure
conn = sqlite3.connect(db_path)

print("Analyzing entity code structure by first digit...")
query = """
SELECT 
    SUBSTR(entity_code, 1, 1) as first_digit,
    COUNT(*) as record_count,
    COUNT(DISTINCT entity_name) as unique_entities
FROM tax_rates 
GROUP BY first_digit 
ORDER BY first_digit
"""
entity_analysis = pd.read_sql_query(query, conn)
print("Entity code distribution by first digit:")
display(entity_analysis)

print("\nEntity code meanings (from Utah State Tax Commission):")
print("1xxx = Counties")
print("2xxx = School districts") 
print("3xxx = Incorporated cities/towns")
print("4xxx = Special service districts")
print("6xxx = Special districts")
print("8xxx/9xxx = Redevelopment or community development areas")

print("\nSample entities by first digit:")
for digit in ['1', '2', '3', '4', '6']:
    query = f"""
    SELECT 
        entity_name,
        entity_code,
        COUNT(*) as frequency
    FROM tax_rates 
    WHERE SUBSTR(entity_code, 1, 1) = '{digit}'
    GROUP BY entity_name, entity_code
    ORDER BY frequency DESC
    LIMIT 5
    """
    result = pd.read_sql_query(query, conn)
    print(f"\nFirst digit {digit}:")
    display(result)

conn.close()

Analyzing entity code structure by first digit...
Entity code distribution by first digit:


,first_digit,record_count,unique_entities
0,1,5178,31
1,2,1726,40
2,3,1249,255
3,4,5327,191
4,5,53,27
5,6,910,12



Entity code meanings (from Utah State Tax Commission):
1xxx = Counties
2xxx = School districts
3xxx = Incorporated cities/towns
4xxx = Special service districts
6xxx = Special districts
8xxx/9xxx = Redevelopment or community development areas

Sample entities by first digit:

First digit 1:


,entity_name,entity_code,frequency
0,COUNTY ASSESSING,1020,1726
1,MULTICOUNTY ASSESSING,1015,1726
2,SALT LAKE,1010,406
3,WEBER,1010,282
4,UTAH,1010,151



First digit 2:


...

,entity_name,entity_code,frequency
0,SANDY CITY,3080,56
1,SALT LAKE CITY,3070,40
2,SALT LAKE LIBRARY,3071,40
3,WEST VALLEY CITY,3120,36
4,WEST JORDAN CITY,3110,33



First digit 4:


,entity_name,entity_code,frequency
0,WEBER BASIN,4005,467
1,CENTRAL UTAH,4185,406
2,SOUTH SALT LAKE VALLEY,4040,321
3,WEBER COUNTY,4080,282
4,JORDAN VALLEY,4045,202



First digit 6:


,entity_name,entity_code,frequency
0,SALT,6030,339
1,ANIMAL WELFARE SERVICES,6000,128
2,COUNTY LIBRARY,6030,128
3,MUNICIPAL TYPE SERVICES,6090,76
4,BOX,6010,57


In [ ]:
# Review official nomenclature PDF for entity code validation
from pdfminer.high_level import extract_text

print("Parsing Weber County nomenclature PDF for entity code validation...")

weber_pdf_path = data_dir / 'weber_nomenclature.pdf'

text = extract_text(weber_pdf_path)
lines = [l.strip() for l in text.split('\n') if l.strip()]

entity_codes = []
for line in lines:
    match = re.match(r'^(\d{4})\s+(.+)$', line)
    if match:
        entity_codes.append({
            'code': match.group(1),
            'name': match.group(2),
        })

print(f"Extracted {len(entity_codes)} entity codes from nomenclature")

nomenclature_df = pd.DataFrame(entity_codes)
print("\nSample entity codes from nomenclature:")
display(nomenclature_df.head(20))

print("\nEntity code distribution in nomenclature:")
nomenclature_df['first_digit'] = nomenclature_df['code'].str[0]
digit_counts = nomenclature_df.groupby('first_digit').size()
print(digit_counts)

print("\nComparing with parsed tax data...")
conn = sqlite3.connect(db_path)
weber_tax_df = pd.read_sql_query(
    "SELECT DISTINCT entity_code, entity_name FROM tax_rates WHERE county = 'WEBER COUNTY' ORDER BY entity_code",
    conn
)
conn.close()

print(f"\nTax data has {len(weber_tax_df)} unique Weber entities")
print(f"Nomenclature has {len(nomenclature_df)} entities")

nomenclature_codes = set(nomenclature_df['code'])
tax_codes = set(weber_tax_df['entity_code'])

missing_in_tax = nomenclature_codes - tax_codes
missing_in_nomenclature = tax_codes - nomenclature_codes

print(f"\nCodes in nomenclature but not in tax data: {len(missing_in_tax)}")
if missing_in_tax:
    print(f"Examples: {list(missing_in_tax)[:10]}")

print(f"\nCodes in tax data but not in nomenclature: {len(missing_in_nomenclature)}")
if missing_in_nomenclature:
    print(f"Examples: {list(missing_in_nomenclature)[:10]}")

Parsing Weber County nomenclature PDF for entity code validation...
PDF has 13 pages

Extracted 120 entity codes from nomenclature

Sample entity codes from nomenclature:


,code,name,page
0,1010,WEBER,1
1,2010,OGDEN CITY SCHOOL DISTRICT,1
2,2020,WEBER COUNTY SCHOOL DISTRICT,1
3,3010,FARR WEST CITY,1
4,3020,HARRISVILLE,1
5,3025,HOOPER,1
6,3030,HUNTSVILLE,1
7,3035,MARRIOTT-SLATERVILLE CITY,1
8,3040,NORTH OGDEN CITY,1
9,3050,OGDEN CITY,1



Entity code distribution in nomenclature:
first_digit
1     1
2     2
3    15
4    37
5     9
6     8
8    42
9     6
dtype: int64

Comparing with parsed tax data...

Tax data has 49 unique Weber entities
Nomenclature has 120 entities

Codes in nomenclature but not in tax data: 73
Examples: ['8011', '4110', '8401', '9701', '8154', '8032', '8029', '8603', '9702', '8105']

Codes in tax data but not in nomenclature: 2
Examples: ['1020', '1015']


In [41]:
# Approach for comparing city-level tax burden (Approach 2)
#
# Goal: Compare the total property tax burden attributable to city-level decisions,
# to enable apples-to-apples comparison between cities.
#
# The problem with using just the 3xxx (city) rate:
#   Some cities provide fire, police, parks, etc. directly (funded in 3xxx).
#   Others outsource those services to special districts (4xxx/6xxx).
#   So 3xxx alone drastically understates the burden for cities that use districts.
#   e.g., West Haven CITY rate = 0.0 but pays 0.001083 to Weber Fire District (4xxx).
#
# Approach 2: total rate - 1xxx (county) - 2xxx (school)
#   County and school rates are set county-wide and identical for all cities in
#   a county. Subtracting them leaves only what varies city by city.
#   This captures: city general fund, fire, police, parks, water, sewer,
#   cemeteries, mosquito abatement, libraries, etc. — everything a city-level
#   decision shapes.
#
# Key filtering rule: only use base tax areas (suffix '- 0000').
#   RDA/CDA overlay areas (suffix '- 9900', '- 9902', etc.) apply only to
#   specific parcels inside redevelopment districts and would inflate the total.
#
# Also note: the raw tax_rates table has duplicate rows for tax areas that span
#   multiple PDF pages. Use DISTINCT on (county, tax_area, entity_code, entity_name,
#   adopted_rate) before summing to avoid double-counting.

conn = sqlite3.connect(db_path)

query = """
WITH deduped AS (
    SELECT DISTINCT county, tax_area, entity_code, entity_name, adopted_rate
    FROM tax_rates
    WHERE tax_area LIKE '% - 0000'
),
city_primary_area AS (
    -- One representative tax area per city: the base area where the city levy appears
    SELECT county, entity_name AS city_name, entity_code AS city_code,
           MIN(tax_area) AS primary_area
    FROM deduped
    WHERE SUBSTR(entity_code, 1, 1) = '3'
    GROUP BY county, entity_name, entity_code
),
breakdown AS (
    SELECT cpa.county, cpa.city_name, cpa.city_code,
        SUM(d.adopted_rate)                                                              AS grand_total,
        SUM(CASE WHEN SUBSTR(d.entity_code,1,1) IN ('1','2') THEN d.adopted_rate ELSE 0 END) AS county_school,
        SUM(CASE WHEN SUBSTR(d.entity_code,1,1) NOT IN ('1','2') THEN d.adopted_rate ELSE 0 END) AS city_burden,
        SUM(CASE WHEN SUBSTR(d.entity_code,1,1) = '3' THEN d.adopted_rate ELSE 0 END)  AS rate_3xxx,
        SUM(CASE WHEN SUBSTR(d.entity_code,1,1) IN ('4','5','6') THEN d.adopted_rate ELSE 0 END) AS rate_districts
    FROM deduped d
    JOIN city_primary_area cpa ON d.county = cpa.county AND d.tax_area = cpa.primary_area
    GROUP BY cpa.county, cpa.city_name, cpa.city_code
)
SELECT county, city_name,
    ROUND(rate_3xxx, 6)      AS city_general_3xxx,
    ROUND(rate_districts, 6) AS local_districts_456,
    ROUND(city_burden, 6)    AS city_burden_approach2,
    ROUND(county_school, 6)  AS county_school_excluded,
    ROUND(grand_total, 6)    AS grand_total
FROM breakdown
ORDER BY county, city_burden DESC
"""

city_burden_df = pd.read_sql_query(query, conn)

print("City-level tax burden by county (Approach 2: total minus county and school)")
print("Only base tax areas (- 0000) used; rows deduplicated across PDF pages.")
print()

for county in city_burden_df['county'].unique():
    county_df = city_burden_df[city_burden_df['county'] == county].reset_index(drop=True)
    print(f"\n{'='*60}")
    print(f"{county}")
    print(f"{'='*60}")
    display(county_df[['city_name','city_general_3xxx','local_districts_456','city_burden_approach2']].to_string(index=False))

conn.close()

City-level tax burden by county (Approach 2: total minus county and school)
Only base tax areas (- 0000) used; rows deduplicated across PDF pages.


BEAVER COUNTY


'       city_name  city_general_3xxx  local_districts_456  city_burden_approach2\n    MILFORD CITY           0.001120             0.001444               0.002564\nMINERSVILLE TOWN           0.000663             0.001444               0.002107\n     BEAVER CITY           0.000170             0.000375               0.000545'


BOX ELDER COUNTY


'      city_name  city_general_3xxx  local_districts_456  city_burden_approach2\n   GARLAND CITY           0.002720             0.000491               0.003211\n TREMONTON CITY           0.002797             0.000397               0.003194\n    MANTUA TOWN           0.001876             0.000493               0.002369\n     PERRY CITY           0.001801             0.000493               0.002294\n   BRIGHAM CITY           0.001685             0.000397               0.002082\n   CORINNE CITY           0.001396             0.000604               0.002000\n   WILLARD CITY           0.001258             0.000588               0.001846\n   PORTAGE TOWN           0.000650             0.000855               0.001505\nDEWEYVILLE TOWN           0.000708             0.000493               0.001201\n SNOWVILLE TOWN           0.000706             0.000493               0.001199\n    ELWOOD TOWN           0.000675             0.000493               0.001168\n    HOWELL CITY           0.000577     


CACHE COUNTY


...

'        city_name  city_general_3xxx  local_districts_456  city_burden_approach2\n   ROCKVILLE TOWN           0.001175             0.001416               0.002591\n TOQUERVILLE CITY           0.000962             0.001416               0.002378\n   LA VERKIN CITY           0.000877             0.001416               0.002293\n   HURRICANE CITY           0.000816             0.001416               0.002232\n      VIRGIN TOWN           0.000422             0.001416               0.001838\n       LEEDS TOWN           0.000418             0.001416               0.001834\n       IVINS CITY           0.001174             0.000420               0.001594\n  SPRINGDALE TOWN           0.000165             0.001416               0.001581\n  ENTERPRISE CITY           0.001113             0.000420               0.001533\n     HILDALE CITY           0.001111             0.000420               0.001531\n NEW HARMONY TOWN           0.000629             0.000865               0.001494\n SANTA CLARA CI


WAYNE COUNTY


'      city_name  city_general_3xxx  local_districts_456  city_burden_approach2\nHANKSVILLE TOWN           0.000375             0.000015               0.000390\n     LYMAN TOWN           0.000181             0.000015               0.000196\n    TORREY TOWN           0.000132             0.000015               0.000147\n  BICKNELL TOWN           0.000118             0.000015               0.000133\n       LOA TOWN           0.000117             0.000015               0.000132'


WEBER COUNTY


'                city_name  city_general_3xxx  local_districts_456  city_burden_approach2\n         HARRISVILLE CITY           0.001534             0.002073               0.003607\n         SOUTH OGDEN CITY           0.002455             0.000795               0.003250\n               OGDEN CITY           0.002193             0.001030               0.003223\n         NORTH OGDEN CITY           0.001304             0.001388               0.002692\n  WASHINGTON TERRACE CITY           0.001667             0.000795               0.002462\n           RIVERDALE CITY           0.001414             0.000795               0.002209\n           FARR WEST CITY           0.000327             0.001878               0.002205\n                 ROY CITY           0.001618             0.000575               0.002193\n       PLEASANT VIEW CITY           0.000767             0.001388               0.002155\n          HUNTSVILLE TOWN           0.000610             0.001338               0.001948\n         

In [42]:
# Query the database for sample data
conn = sqlite3.connect(db_path)

# Get first 20 rows
query = "SELECT * FROM tax_rates LIMIT 20"
result = pd.read_sql_query(query, conn)
print("First 20 records:")
display(result)

# Get top 10 highest tax rates
query = """
SELECT county, entity_name, adopted_rate
FROM tax_rates
ORDER BY adopted_rate DESC
LIMIT 10
"""
top_rates = pd.read_sql_query(query, conn)
print("\nTop 10 highest tax rates:")
display(top_rates)

conn.close()

First 20 records:


,county,tax_area,entity_code,entity_name,auditor_rate,certified_rate,adopted_rate,page
0,BEAVER COUNTY,001 - 0000,1010,BEAVER,0.001462,0.001462,0.001462,1
1,BEAVER COUNTY,001 - 0000,1015,MULTICOUNTY ASSESSING,0.000014,0.000014,0.000014,1
2,BEAVER COUNTY,001 - 0000,1020,COUNTY ASSESSING,0.000294,0.000294,0.000294,1
3,BEAVER COUNTY,001 - 0000,2010,BEAVER,0.005190,0.005190,0.005190,1
4,BEAVER COUNTY,001 - 0000,4010,BEAVER COUNTY SPECIAL,0.000375,0.000375,0.000375,1
5,BEAVER COUNTY,001 - 0005,1010,BEAVER,0.001462,0.001462,0.001462,1
6,BEAVER COUNTY,001 - 0005,1015,MULTICOUNTY ASSESSING,0.000014,0.000014,0.000014,1
7,BEAVER COUNTY,001 - 0005,1020,COUNTY ASSESSING,0.000294,0.000294,0.000294,1
8,BEAVER COUNTY,001 - 0005,2010,BEAVER,0.005190,0.005190,0.005190,1
9,BEAVER COUNTY,001 - 0005,4010,BEAVER COUNTY SPECIAL,0.000375,0.000375,0.000375,1



Top 10 highest tax rates:


,county,entity_name,adopted_rate
0,UTAH COUNTY,MEDICAL SCHOOL CAMPUS,0.015000
1,UTAH COUNTY,MEDICAL SCHOOL CAMPUS,0.015000
2,SALT LAKE COUNTY,AUTO-MALL,0.010000
3,SALT LAKE COUNTY,AUTO-MALL,0.010000
4,WASHINGTON COUNTY,BLACK DESERT,0.010000
5,WASHINGTON COUNTY,JEPPSON CANYON PUBLIC INFRASTRUCTURE,0.009000
6,CARBON COUNTY,EAST,0.008750
7,EMERY COUNTY,EMERY,0.008671
8,EMERY COUNTY,EMERY,0.008671
9,EMERY COUNTY,EMERY,0.008671
